# YOLO + SAM — Model Download & Testing

This notebook installs the required packages, loads YOLO and SAM, and tests detection plus point/box-prompted segmentation.

In [ ]:
!pip install -q -U ultralytics opencv-python pillow numpy

In [ ]:
from ultralytics import YOLO, SAM
import cv2
import numpy as np
import matplotlib.pyplot as plt

YOLO_WEIGHTS = 'yolo26n.pt'
SAM_WEIGHTS = 'sam_b.pt'

yolo = YOLO(YOLO_WEIGHTS)
sam = SAM(SAM_WEIGHTS)
print('YOLO and SAM loaded successfully.')

## Download/test image

In [ ]:
import urllib.request
url = 'https://ultralytics.com/images/bus.jpg'
image_path = 'bus.jpg'
urllib.request.urlretrieve(url, image_path)

image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 6))
plt.imshow(image_rgb)
plt.axis('off');

## 1. YOLO object detection

In [ ]:
yolo_results = yolo.predict(source=image, conf=0.25, verbose=False)
detected = yolo_results[0].plot()

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(detected, cv2.COLOR_BGR2RGB))
plt.axis('off');

## 2. SAM segmentation with a point prompt

The point below is only an example. Change `(500, 375)` to a point inside the object you want to segment.

In [ ]:
point = [[500, 375]]
point_labels = [1]  # 1 = foreground
point_results = sam.predict(
    source=image,
    points=point,
    labels=point_labels,
    verbose=False,
)

point_result = point_results[0]
point_result.show()

## 3. SAM segmentation with a bounding box prompt

Use XYXY coordinates: `[x1, y1, x2, y2]`.

In [ ]:
box = [[100, 100, 600, 500]]
box_results = sam.predict(
    source=image,
    bboxes=box,
    verbose=False,
)

box_result = box_results[0]
box_result.show()

## 4. Extract the mask as a NumPy array

In [ ]:
if point_result.masks is not None:
    mask = point_result.masks.data[0].cpu().numpy().astype(np.uint8)
    print('Mask shape:', mask.shape)
    print('Mask pixels:', int(mask.sum()))
else:
    print('No mask returned.')

## 5. Optional: save the mask

In [ ]:
if point_result.masks is not None:
    cv2.imwrite('sam_point_mask.png', mask * 255)
    print('Saved: sam_point_mask.png')